In [4]:
import numpy as np
import pandas as pd

#for reproductiblity
np.random.seed(42)
#create sample data
num_rows = 100

df = pd.DataFrame({
    'customer_id': range(1001, 1001 + num_rows),
    'age': np.random.randint(18, 65, num_rows),
    'salary' : np.random.randint(50000, 150000, num_rows),
    'city' : np.random.choice(
        ['Delhi', 'Mumbai', 'Bengluru', 'chennai'], num_rows),
    'experience': np.random.randint(1, 15, num_rows)
})


In [5]:
df

,customer_id,age,salary,city,experience
0,1001,56,52695,Mumbai,1
1,1002,46,98190,Delhi,2
2,1003,32,55258,Mumbai,1
3,1004,60,137538,chennai,14
4,1005,25,89504,chennai,12
...,...,...,...,...,...
95,1096,24,143557,Mumbai,9
96,1097,26,111087,chennai,5
97,1098,41,145839,Delhi,8
98,1099,18,118840,chennai,13


In [8]:
# INTRODUCE MISSING VALUES RANDOMLY
# Added the missing colon (:) at the end of the line
for col in ['age', 'salary', 'city']:
    # Fixed the typo "idex" to "index" and indented the loop block
    missing_indices = np.random.choice(df.index, size=5, replace=False)
    df.loc[missing_indices, col] = np.nan

# Introduce duplicate rows
duplicate_rows = df.sample(3)
df = pd.concat([df, duplicate_rows], ignore_index=True) # Added ignore_index=True to prevent duplicate index bugs

# Introduce outliers
# Now that the index is unique, this will safely pick exactly 2 rows
outlier_indices = np.random.choice(df.index, size=2, replace=False)
df.loc[outlier_indices, 'salary'] = [9999999, 8888888]

# View the result
print(df.tail(10)) # Using tail() so you can easily see the newly appended duplicate rows!

     customer_id   age    salary     city  experience
93          1094  18.0   84766.0  chennai           3
94          1095  42.0       NaN  chennai          12
95          1096   NaN  143557.0   Mumbai           9
96          1097  26.0  111087.0  chennai           5
97          1098  41.0  145839.0    Delhi           8
98          1099  18.0  118840.0  chennai          13
99          1100  61.0  104384.0  chennai           1
100         1004  60.0  137538.0  chennai          14
101         1062  21.0   99080.0  chennai           6
102         1078  31.0   80080.0  chennai           1


Data cleaning

In [9]:
df.isna().sum() # gives total missing values in data

,0
customer_id,0
age,5
salary,5
city,5
experience,0


Data Imputation
-Numeric Type - Replace with either mean or median[if outliers are present in your data we prefer to replace the missing values with median]
-Catergorical Type - Replace with the mode

In [11]:
num_cols = df.select_dtypes(include = np.number).columns
num_cols

Index(['customer_id', 'age', 'salary', 'experience'], dtype='object')

In [17]:
# Assuming you want to target numeric columns, you can define num_cols like this:
num_cols = ['age', 'salary', 'experience']

# Loop to fill missing values with the median
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Check if any missing values remain (Moved OUTSIDE the loop)
print(df.isna().sum())

customer_id    0
age            0
salary         0
city           5
experience     0
dtype: int64


In [18]:
# filter all the categorical columns
cat_cols = df.select_dtypes(include = 'object').columns
cat_cols

Index(['city'], dtype='object')

In [20]:
for col in cat_cols:
  df[col] = df[col].fillna(df[col].mode()[0])

In [21]:
df.isna().sum()

,0
customer_id,0
age,0
salary,0
city,0
experience,0


#finding outliers

In [26]:
for col in num_cols:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR

  df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound) | df[col].isna()]
  print("Final dataset shape:", df.shape)

Final dataset shape: (101, 5)
Final dataset shape: (101, 5)
Final dataset shape: (101, 5)
